In [9]:
# from pipeline_stable_diffusion_xl_scope import SCoPEDiffusionXLPipeline
# from diffusers import DiffusionPipeline as SCoPEDiffusionXLPipeline
from pipeline_stable_diffusion_xl import StableDiffusionXLPipeline as SCoPEDiffusionXLPipeline
import torch

# load both base & refiner
base = SCoPEDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0", torch_dtype=torch.float16, variant="fp16", use_safetensors=True,
    cache_dir = '/projectnb/vkolagrp/ketanss/scope-diffusers/sdpcache'
)

base.to("cuda")
refiner = SCoPEDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-refiner-1.0",
    text_encoder_2=base.text_encoder_2,
    vae=base.vae,
    torch_dtype=torch.float16,
    use_safetensors=True,
    variant="fp16",
)
refiner.to("cuda")



ImportError: attempted relative import with no known parent package

In [2]:
# Define how many steps and what % of steps to be run on each experts (80/20) here
n_steps = 40
high_noise_frac = 0.8

prompt = "A majestic lion jumping from a big stone at night"

# run both experts
image = base(
    prompt=prompt,
    num_inference_steps=n_steps,
    denoising_end=high_noise_frac,
    output_type="latent",
).images


  0%|          | 0/32 [00:00<?, ?it/s]

In [3]:
image = refiner(
    prompt=prompt,
    num_inference_steps=n_steps,
    denoising_start=high_noise_frac,
    image=image,
).images[0]
image

ValueError: Model expects an added time embedding vector of length 2560, but a vector of 2816 was created. The model has an incorrect config. Please check `unet.config.time_embedding_type` and `text_encoder_2.config.projection_dim`.